# NBA Breakout Analysis
**Author:** Mike Zhang  
**Date:** 2026-03-19  
**Purpose:** Identify players who broke out in the second half of seasons on non-playoff teams and examine whether their post all-star performance from the previous season predicts their following season performance

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nba_api.stats.endpoints import leaguedashplayerstats

In [3]:
# Pull post all-star stats for 2023-24
post_allstar = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2023-24',
    season_segment_nullable='Post All-Star',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

# Pull pre all-star stats for 2023-24
pre_allstar = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2023-24',
    season_segment_nullable='Pre All-Star',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

# Pull following full season stats (2024-25)
following_season = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2024-25',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

# Pull prior full season stats (2022-23)
prior_season = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2022-23',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

print("Post all-star shape:", post_allstar.shape)
print("Pre all-star shape:", pre_allstar.shape)
print("Following season shape:", following_season.shape)
print("Prior season shape:", prior_season.shape)

Post all-star shape: (518, 79)
Pre all-star shape: (540, 79)
Following season shape: (569, 79)
Prior season shape: (539, 79)


In [22]:
# Columns to keep from pre and post all-star splits
split_cols = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 
              'GP', 'MIN', 'PIE']

# Columns to keep from following season
following_cols = ['PLAYER_ID', 'GP', 'PIE']
prior_cols = ['PLAYER_ID', 'PIE']

# Trim each dataframe
post = post_allstar[split_cols].copy()
pre = pre_allstar[split_cols].copy()
following = following_season[following_cols].copy()
prior = prior_season[prior_cols].copy()

print(post.head())
print(pre.head())
print(following.head())
print(prior.head())

   PLAYER_ID    PLAYER_NAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP   MIN  \
0    1630639    A.J. Lawson  1610612742               DAL  23.0  15   5.4   
1    1631260       AJ Green  1610612749               MIL  24.0  20  13.9   
2    1631100     AJ Griffin  1610612737               ATL  20.0   2  19.1   
3     203932   Aaron Gordon  1610612743               DEN  28.0  24  30.9   
4    1628988  Aaron Holiday  1610612745               HOU  27.0  27  13.2   

     PIE  
0  0.081  
1  0.059  
2  0.014  
3  0.110  
4  0.066  
   PLAYER_ID    PLAYER_NAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP   MIN  \
0    1630639    A.J. Lawson  1610612742               DAL  23.0  27   8.5   
1    1631260       AJ Green  1610612749               MIL  24.0  36   9.3   
2    1631100     AJ Griffin  1610612737               ATL  20.0  18   7.4   
3     203932   Aaron Gordon  1610612743               DEN  28.0  49  31.7   
4    1628988  Aaron Holiday  1610612745               HOU  27.0  51  17.9   

     PIE

In [23]:
# Step 1: Apply post all-star inclusion criteria
# - At least 15 games played
# - At least 25 MPG
# - Age 28 or younger

post_filtered = post[
    (post['GP'] >= 15) & 
    (post['MIN'] >= 25) &
    (post['AGE'] <= 28)
].copy()

# Step 2: Apply pre all-star inclusion criteria
# - At least 20 games played
# - At least 10 MPG

pre_filtered = pre[
    (pre['GP'] >= 20) & 
    (pre['MIN'] >= 10)
].copy()

following_filtered = following[
    (following['GP'] >= 30)
].copy()

print("Post all-star candidates:", len(post_filtered))
print("Pre all-star candidates:", len(pre_filtered))
print("Following season candidates:", len(following_filtered))

Post all-star candidates: 109
Pre all-star candidates: 355
Following season candidates: 411


In [27]:
# Merge pre and post all-star data on PLAYER_ID
merged = post_filtered.merge(
    pre_filtered,
    on='PLAYER_ID',
    suffixes=('_post', '_pre')
)

# Merge filtered following season PIE data on PLAYER_ID
merged = merged.merge(
    following_filtered[['PLAYER_ID', 'PIE']].rename(columns={'PIE': 'PIE_following'}),
    on='PLAYER_ID',
    how='left'
)

# Apply 40% minutes jump criterion
merged = merged[
    (merged['PIE_post'] - merged['PIE_pre'] >= 0.02) | (merged['MIN_post'] >= 1.3 * merged['MIN_pre'])
].copy()
print("Players meeting all criteria:", len(merged))
print(merged[['PLAYER_NAME_post', 'MIN_pre', 'MIN_post', 'PIE_post', 'PIE_pre']].to_string())

Players meeting all criteria: 28
     PLAYER_NAME_post  MIN_pre  MIN_post  PIE_post  PIE_pre
2       Amen Thompson     19.2      26.5     0.134    0.107
3         Amir Coffey     17.9      25.0     0.050    0.067
8         Ayo Dosunmu     25.5      37.5     0.092    0.077
13    Cade Cunningham     34.0      32.1     0.148    0.117
20      Corey Kispert     22.5      32.4     0.082    0.085
23       De'Aaron Fox     35.6      36.5     0.143    0.121
25      Deandre Ayton     31.7      34.0     0.175    0.124
26    Dejounte Murray     35.0      37.1     0.138    0.114
32   Donte DiVincenzo     25.0      37.5     0.094    0.101
35         GG Jackson     19.4      31.4     0.090    0.105
36     Gary Trent Jr.     26.3      32.4     0.099    0.066
37        Gradey Dick     15.3      28.8     0.060    0.057
38     Grant Williams     26.7      30.7     0.085    0.053
41  Immanuel Quickley     27.2      34.7     0.138    0.109
49      Jalen Brunson     35.9      34.5     0.177    0.147
52     

In [28]:
# Filter for "tanking" teams in post all-star data
from nba_api.stats.endpoints import leaguedashteamstats

pre_allstar_teams = leaguedashteamstats.LeagueDashTeamStats(
    season='2023-24',
    season_segment_nullable='Pre All-Star'
).get_data_frames()[0]

print(pre_allstar_teams[['TEAM_NAME', 'TEAM_ID', 'GP', 'W', 'L', 'W_PCT']].to_string())
tanking_teams = pre_allstar_teams.nsmallest(12, 'W_PCT')[['TEAM_NAME', 'TEAM_ID', 'W_PCT']].copy()
print(tanking_teams.sort_values('W_PCT').to_string())

                 TEAM_NAME     TEAM_ID  GP   W   L  W_PCT
0            Atlanta Hawks  1610612737  55  24  31  0.436
1           Boston Celtics  1610612738  55  43  12  0.782
2            Brooklyn Nets  1610612751  54  21  33  0.389
3        Charlotte Hornets  1610612766  54  13  41  0.241
4            Chicago Bulls  1610612741  55  26  29  0.473
5      Cleveland Cavaliers  1610612739  53  36  17  0.679
6         Dallas Mavericks  1610612742  55  32  23  0.582
7           Denver Nuggets  1610612743  55  36  19  0.655
8          Detroit Pistons  1610612765  54   8  46  0.148
9    Golden State Warriors  1610612744  53  27  26  0.509
10         Houston Rockets  1610612745  54  24  30  0.444
11          Indiana Pacers  1610612754  56  31  25  0.554
12             LA Clippers  1610612746  53  36  17  0.679
13      Los Angeles Lakers  1610612747  56  30  26  0.536
14       Memphis Grizzlies  1610612763  56  20  36  0.357
15              Miami Heat  1610612748  55  30  25  0.545
16         Mil

In [29]:
# Filter merged players to only tanking teams
merged_tanking = merged[
    merged['TEAM_ID_post'].isin(tanking_teams['TEAM_ID'])
].copy()

print("Players on tanking teams:", len(merged_tanking))
print(merged_tanking[['PLAYER_NAME_post', 'TEAM_ID_post', 'MIN_pre', 'MIN_post', 'PIE_pre', 'PIE_post']].to_string())


Players on tanking teams: 17
     PLAYER_NAME_post  TEAM_ID_post  MIN_pre  MIN_post  PIE_pre  PIE_post
2       Amen Thompson    1610612745     19.2      26.5    0.107     0.134
8         Ayo Dosunmu    1610612741     25.5      37.5    0.077     0.092
13    Cade Cunningham    1610612765     34.0      32.1    0.117     0.148
20      Corey Kispert    1610612764     22.5      32.4    0.085     0.082
25      Deandre Ayton    1610612757     31.7      34.0    0.124     0.175
26    Dejounte Murray    1610612737     35.0      37.1    0.114     0.138
35         GG Jackson    1610612763     19.4      31.4    0.105     0.090
36     Gary Trent Jr.    1610612761     26.3      32.4    0.066     0.099
37        Gradey Dick    1610612761     15.3      28.8    0.057     0.060
38     Grant Williams    1610612766     26.7      30.7    0.053     0.085
41  Immanuel Quickley    1610612761     27.2      34.7    0.109     0.138
52      Jalen Johnson    1610612737     34.0      33.0    0.112     0.134
61       

In [9]:
merged_prior = merged_tanking.merge(
    prior.rename(columns={'PIE': 'PIE_previous'}),
    on='PLAYER_ID',
    how='left'
) 

merged_prior_filtered = merged_prior[
    (merged_prior['PIE_previous'] <= 0.110) | (merged_prior['PIE_previous'].isna())
].copy()

print(merged_prior_filtered[['PLAYER_NAME_post', 'TEAM_ID_post', 'MIN_pre', 'MIN_post', 'PIE_previous', 'PIE_pre', 'PIE_post']].to_string())


     PLAYER_NAME_post  TEAM_ID_post  MIN_pre  MIN_post  PIE_previous  PIE_pre  PIE_post
0       Amen Thompson    1610612745     19.2      26.5           NaN    0.107     0.134
1         Ayo Dosunmu    1610612741     25.5      37.5         0.065    0.077     0.092
2     Cade Cunningham    1610612765     34.0      32.1         0.109    0.117     0.148
3       Corey Kispert    1610612764     22.5      32.4         0.073    0.085     0.082
6          GG Jackson    1610612763     19.4      31.4           NaN    0.105     0.090
7      Gary Trent Jr.    1610612761     26.3      32.4         0.091    0.066     0.099
8         Gradey Dick    1610612761     15.3      28.8           NaN    0.057     0.060
9      Grant Williams    1610612766     26.7      30.7         0.069    0.053     0.085
11      Jalen Johnson    1610612737     34.0      33.0         0.099    0.112     0.134
12       John Collins    1610612762     27.8      28.4         0.086    0.104     0.134
13     Jordan Goodwin    1610612

In [32]:
# Merge following season net rating
final = merged_prior_filtered.merge(
    following_filtered[['PLAYER_ID', 'PIE']].rename(columns={'PIE': 'PIE_following'}),
    on='PLAYER_ID',
    how='left'
)

print(final[['PLAYER_NAME_post', 'PIE_previous', 'PIE_post', 'PIE_pre', 'PIE_following']].to_string())

     PLAYER_NAME_post  PIE_previous  PIE_post  PIE_pre  PIE_following
0       Amen Thompson           NaN     0.134    0.107          0.129
1         Ayo Dosunmu         0.065     0.092    0.077          0.084
2     Cade Cunningham         0.109     0.148    0.117          0.152
3       Corey Kispert         0.073     0.082    0.085          0.074
4          GG Jackson           NaN     0.090    0.105            NaN
5      Gary Trent Jr.         0.091     0.099    0.066          0.070
6         Gradey Dick           NaN     0.060    0.057          0.071
7      Grant Williams         0.069     0.085    0.053            NaN
8       Jalen Johnson         0.099     0.134    0.112          0.135
9        John Collins         0.086     0.134    0.104          0.129
10     Jordan Goodwin         0.102     0.100    0.095            NaN
11       Jordan Poole         0.104     0.106    0.061          0.107
12  Julian Champagnie         0.086     0.070    0.066          0.079
13        Kris Murra

In [36]:
def get_breakout_candidates(season):
    # Construct season strings
    start_year = int(season.split('-')[0])
    following_season_str = f"{start_year + 1}-{str(start_year + 2)[-2:]}"
    previous_season_str = f"{start_year - 1}-{str(start_year)[-2:]}"
    
    # Pull post all-star player stats
    post = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_segment_nullable='Post All-Star',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull pre all-star player stats
    pre = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_segment_nullable='Pre All-Star',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull following full season stats
    following = leaguedashplayerstats.LeagueDashPlayerStats(
        season=following_season_str,
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull previous full season stats
    previous = leaguedashplayerstats.LeagueDashPlayerStats(
        season=previous_season_str,
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull pre all-star team standings
    pre_allstar_teams = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_segment_nullable='Pre All-Star'
    ).get_data_frames()[0]
    
    # Identify tanking teams (bottom 12 by pre all-star win rate)
    tanking_teams = pre_allstar_teams.nsmallest(12, 'W_PCT')[['TEAM_ID', 'TEAM_NAME']]
    
    # Keep relevant columns
    split_cols = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 
                  'AGE', 'GP', 'MIN', 'PIE', 'USG_PCT']
    post = post[split_cols].copy()
    pre = pre[split_cols].copy()
    following = following[['PLAYER_ID', 'PIE', 'GP']].copy()
    previous = previous[['PLAYER_ID', 'PIE']].copy()
    
    # Apply post all-star filters
    post_filtered = post[
        (post['GP'] >= 15) &
        (post['MIN'] >= 25) &
        (post['AGE'] <= 28)
    ].copy()
    
    # Apply pre all-star filters
    pre_filtered = pre[
        (pre['GP'] >= 20) &
        (pre['MIN'] >= 10)
    ].copy()
    
    # Merge pre and post on PLAYER_ID
    merged = post_filtered.merge(
        pre_filtered,
        on='PLAYER_ID',
        suffixes=('_post', '_pre')
    )
    
    # Apply breakout criterion
    merged = merged[
        (merged['PIE_post'] - merged['PIE_pre'] >= 0.02) |
        (merged['MIN_post'] >= 1.3 * merged['MIN_pre'])
    ].copy()
    
    # Filter for tanking teams
    merged = merged[
        merged['TEAM_ID_post'].isin(tanking_teams['TEAM_ID'])
    ].copy()
    
    # Merge previous season PIE and filter out established players
    merged = merged.merge(
        previous[['PLAYER_ID', 'PIE']].rename(columns={'PIE': 'PIE_previous'}),
        on='PLAYER_ID',
        how='left'
    )
    
    # Keep rookies (no previous season data) or players with PIE <= 0.110
    merged = merged[
        (merged['PIE_previous'].isna()) |
        (merged['PIE_previous'] <= 0.110)
    ].copy()
    
    # Merge following season PIE and GP
    merged = merged.merge(
        following[['PLAYER_ID', 'PIE', 'GP']].rename(columns={
            'PIE': 'PIE_following',
            'GP': 'GP_following'
        }),
        on='PLAYER_ID',
        how='left'
    )
    
    # Filter for minimum games played in following season
    merged = merged[
        merged['GP_following'] >= 30
    ].copy()
    
    # Add season label
    merged['season'] = season
    
    return merged

print("Function defined successfully")

Function defined successfully


In [39]:
test = get_breakout_candidates('2022-23')
print(test.shape)
print(test[['PLAYER_NAME_post','MIN_pre', 'MIN_post', 'PIE_post', 'PIE_pre', 'PIE_previous', 'PIE_following']].to_string())

(14, 21)
       PLAYER_NAME_post  MIN_pre  MIN_post  PIE_post  PIE_pre  PIE_previous  PIE_following
0       Andrew Nembhard     27.1      29.0     0.085    0.060           NaN          0.075
1         Austin Reaves     27.9      30.4     0.138    0.077         0.076          0.105
2            Coby White     22.3      25.9     0.110    0.076         0.080          0.099
3      Dennis Smith Jr.     25.0      27.4     0.105    0.084         0.095          0.092
6         James Wiseman     13.0      25.3     0.103    0.107           NaN          0.097
7          Jordan Nwora     15.6      25.7     0.099    0.078         0.072          0.103
8      Matisse Thybulle     12.7      27.6     0.057    0.052         0.057          0.061
9          Ochai Agbaji     15.1      29.6     0.058    0.045           NaN          0.046
10       Shaedon Sharpe     20.3      27.0     0.085    0.063           NaN          0.080
11  Talen Horton-Tucker     16.5      29.1     0.110    0.092         0.070      

In [62]:
import time

seasons = [
    '2014-15', '2015-16', '2016-17', '2017-18', '2018-19',
    '2019-20', '2020-21', '2021-22', '2022-23', '2023-24'
]

all_seasons = []

for season in seasons:
    print(f"Processing {season}...")
    try:
        df = get_breakout_candidates(season)
        all_seasons.append(df)
        print(f"{season}: {len(df)} candidates found")
    except Exception as e:
        print(f"{season}: Error - {e}")
    time.sleep(3)  # pause between API calls to avoid rate limiting

# Combine all seasons
all_candidates = pd.concat(all_seasons, ignore_index=True)
all_candidates = all_candidates[all_candidates['season'] != '2019-20'].copy()
print(f"\nTotal candidates across all seasons: {len(all_candidates)}")

Processing 2014-15...
2014-15: 18 candidates found
Processing 2015-16...
2015-16: 5 candidates found
Processing 2016-17...
2016-17: 9 candidates found
Processing 2017-18...
2017-18: 7 candidates found
Processing 2018-19...
2018-19: 8 candidates found
Processing 2019-20...
2019-20: 2 candidates found
Processing 2020-21...
2020-21: 12 candidates found
Processing 2021-22...
2021-22: 16 candidates found
Processing 2022-23...
2022-23: 14 candidates found
Processing 2023-24...
2023-24: 11 candidates found

Total candidates across all seasons: 100


In [ ]:
# Check distribution across seasons
print(all_candidates.groupby('season').size())

# Check for any missing PIE_following values
print("\nMissing PIE_following:", all_candidates['PIE_following'].isna().sum())

# Quick summary stats
print("\nPIE_post summary:")
print(all_candidates['PIE_post'].describe())

print("\nPIE_following summary:")
print(all_candidates['PIE_following'].describe())


season
2014-15    18
2015-16     5
2016-17     9
2017-18     7
2018-19     8
2019-20     2
2020-21    12
2021-22    16
2022-23    14
2023-24    11
dtype: int64

Missing PIE_following: 0

PIE_post summary:
count    102.000000
mean       0.099549
std        0.026224
min        0.052000
25%        0.082000
50%        0.099000
75%        0.117000
max        0.174000
Name: PIE_post, dtype: float64

PIE_following summary:
count    102.000000
mean       0.091471
std        0.022876
min        0.039000
25%        0.074000
50%        0.092000
75%        0.107000
max        0.160000
Name: PIE_following, dtype: float64


In [53]:
from nba_api.stats.endpoints import leaguestandingsv3

def get_team_wins(season):
    standings = leaguestandingsv3.LeagueStandingsV3(
        season=season
    ).get_data_frames()[0]
    
    return standings[['TeamID', 'TeamCity', 'TeamName', 'WINS']].copy()

# Test it
test_wins = get_team_wins('2023-24')
print(test_wins.head())

       TeamID       TeamCity TeamName  WINS
0  1610612738         Boston  Celtics    64
1  1610612760  Oklahoma City  Thunder    57
2  1610612743         Denver  Nuggets    57
3  1610612752       New York   Knicks    50
4  1610612749      Milwaukee    Bucks    49


In [54]:
def get_three_year_avg_wins(season):
    start_year = int(season.split('-')[0])
    
    # Construct 3 preceding season strings
    prior_seasons = [
        f"{start_year - 3}-{str(start_year - 2)[-2:]}",
        f"{start_year - 2}-{str(start_year - 1)[-2:]}",
        f"{start_year - 1}-{str(start_year)[-2:]}"
    ]
    
    dfs = []
    for s in prior_seasons:
        df = get_team_wins(s)
        df['prior_season'] = s
        dfs.append(df)
        time.sleep(1)
    
    # Combine and average wins by team
    combined = pd.concat(dfs, ignore_index=True)
    avg_wins = combined.groupby('TeamID')['WINS'].mean().reset_index()
    avg_wins.columns = ['TEAM_ID', 'avg_wins_3yr']
    
    return avg_wins

# Test on 2023-24
test_avg = get_three_year_avg_wins('2023-24')
print(test_avg.sort_values('avg_wins_3yr', ascending=False).head(10))

       TEAM_ID  avg_wins_3yr
19  1610612756     53.333333
12  1610612749     51.666667
18  1610612755     51.333333
6   1610612743     49.333333
26  1610612763     48.333333
1   1610612738     48.000000
25  1610612762     46.000000
11  1610612748     45.666667
14  1610612751     45.666667
7   1610612744     45.333333


In [55]:
# Merge team names for readability
team_names = get_team_wins('2023-24')[['TeamID', 'TeamCity', 'TeamName']].copy()
team_names.columns = ['TEAM_ID', 'TeamCity', 'TeamName']

test_avg_named = test_avg.merge(team_names, on='TEAM_ID')
test_avg_named['full_name'] = test_avg_named['TeamCity'] + ' ' + test_avg_named['TeamName']

print(test_avg_named[['full_name', 'avg_wins_3yr']]
      .sort_values('avg_wins_3yr', ascending=False)
      .head(10)
      .to_string())

                full_name  avg_wins_3yr
19           Phoenix Suns     53.333333
12        Milwaukee Bucks     51.666667
18     Philadelphia 76ers     51.333333
6          Denver Nuggets     49.333333
26      Memphis Grizzlies     48.333333
1          Boston Celtics     48.000000
25              Utah Jazz     46.000000
11             Miami Heat     45.666667
14          Brooklyn Nets     45.666667
7   Golden State Warriors     45.333333


In [64]:
seasons_no_covid = [s for s in seasons if s != '2019-20']

avg_wins_all = []
for season in seasons_no_covid:
    print(f"Getting wins for {season}...")
    avg_wins = get_three_year_avg_wins(season)
    avg_wins['season'] = season
    avg_wins = avg_wins.rename(columns={'TEAM_ID': 'TEAM_ID_post'})
    avg_wins_all.append(avg_wins)
    time.sleep(1)

avg_wins_combined = pd.concat(avg_wins_all, ignore_index=True)

# Merge into all_candidates
all_candidates = all_candidates.merge(
    avg_wins_combined,
    on=['TEAM_ID_post', 'season'],
    how='left'
)

print("Missing avg_wins_3yr:", all_candidates['avg_wins_3yr'].isna().sum())
print(all_candidates[['PLAYER_NAME_post', 'season', 'avg_wins_3yr']].head(10).to_string())

Getting wins for 2014-15...
Getting wins for 2015-16...
Getting wins for 2016-17...
Getting wins for 2017-18...
Getting wins for 2018-19...
Getting wins for 2020-21...
Getting wins for 2021-22...
Getting wins for 2022-23...
Getting wins for 2023-24...
Missing avg_wins_3yr: 0
   PLAYER_NAME_post   season  avg_wins_3yr
0  Bojan Bogdanovic  2014-15     38.333333
1          CJ Miles  2014-15     49.000000
2    Chase Budinger  2014-15     32.333333
3  Danilo Gallinari  2014-15     43.666667
4     Elfrid Payton  2014-15     26.666667
5     Isaiah Canaan  2014-15     29.333333
6       Jae Crowder  2014-15     35.000000
7        Jeremy Lin  2014-15     37.666667
8   Jordan Clarkson  2014-15     37.666667
9      Nerlens Noel  2014-15     29.333333


In [ ]:
all_candidates['PIE_change'] = all_candidates['PIE_post'] - all_candidates['PIE_pre']
all_candidates['USG_change'] = all_candidates['USG_PCT_post'] - all_candidates['USG_PCT_pre']
all_candidates['PIE_diff'] = all_candidates['PIE_post'] - all_candidates['PIE_following']

print(all_candidates[['PLAYER_NAME_post', 'PIE_change', 'USG_change', 'PIE_diff']].head(10).to_string())

   PLAYER_NAME_post  PIE_change  USG_change  PIE_diff
0  Bojan Bogdanovic       0.032       0.018     0.020
1          CJ Miles       0.027      -0.012     0.025
2    Chase Budinger       0.037       0.006     0.031
3  Danilo Gallinari       0.026       0.030     0.000
4     Elfrid Payton       0.029       0.000     0.017
5     Isaiah Canaan       0.003       0.036     0.015
6       Jae Crowder       0.022       0.040     0.000
7        Jeremy Lin       0.037       0.038     0.032
8   Jordan Clarkson       0.033       0.024     0.034
9      Nerlens Noel       0.051       0.035     0.027


In [69]:
from nba_api.stats.endpoints import commonplayerinfo

# Test on one player to see what's available
test_player = commonplayerinfo.CommonPlayerInfo(
    player_id=all_candidates['PLAYER_ID'].iloc[0]
).get_data_frames()[0]

print(test_player.columns.tolist())
print(test_player[['DISPLAY_FIRST_LAST', 'POSITION']].to_string())

['PERSON_ID', 'FIRST_NAME', 'LAST_NAME', 'DISPLAY_FIRST_LAST', 'DISPLAY_LAST_COMMA_FIRST', 'DISPLAY_FI_LAST', 'PLAYER_SLUG', 'BIRTHDATE', 'SCHOOL', 'COUNTRY', 'LAST_AFFILIATION', 'HEIGHT', 'WEIGHT', 'SEASON_EXP', 'JERSEY', 'POSITION', 'ROSTERSTATUS', 'GAMES_PLAYED_CURRENT_SEASON_FLAG', 'TEAM_ID', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'TEAM_CODE', 'TEAM_CITY', 'PLAYERCODE', 'FROM_YEAR', 'TO_YEAR', 'DLEAGUE_FLAG', 'NBA_FLAG', 'GAMES_PLAYED_FLAG', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GREATEST_75_FLAG']
  DISPLAY_FIRST_LAST POSITION
0   Bojan Bogdanović  Forward


In [70]:
positions = []

for player_id in all_candidates['PLAYER_ID'].unique():
    try:
        info = commonplayerinfo.CommonPlayerInfo(
            player_id=player_id
        ).get_data_frames()[0]
        positions.append({
            'PLAYER_ID': player_id,
            'POSITION': info['POSITION'].iloc[0]
        })
        time.sleep(0.5)
    except Exception as e:
        print(f"Error for player {player_id}: {e}")

positions_df = pd.DataFrame(positions)
print(positions_df['POSITION'].value_counts())

POSITION
Guard             43
Forward           23
Guard-Forward     10
Center             8
Forward-Guard      4
Forward-Center     4
Center-Forward     2
Name: count, dtype: int64


In [73]:
# Simplify positions to three categories
position_map = {
    'Guard': 'Guard',
    'Guard-Forward': 'Guard',
    'Forward-Guard': 'Guard',
    'Forward': 'Forward',
    'Forward-Center': 'Forward',
    'Center-Forward': 'Forward',
    'Center': 'Center'
}

positions_df['POSITION_simplified'] = positions_df['POSITION'].map(position_map)

# Merge into all_candidates
all_candidates = all_candidates.merge(
    positions_df[['PLAYER_ID', 'POSITION_simplified']],
    on='PLAYER_ID',
    how='left'
)

print("Missing positions:", all_candidates['POSITION_simplified'].isna().sum())
print(all_candidates['POSITION_simplified'].value_counts())

Missing positions: 0
POSITION_simplified
Guard      62
Forward    30
Center      8
Name: count, dtype: int64


In [74]:
model_cols = [
    'PLAYER_NAME_post', 'season',
    'PIE_diff',          # primary outcome
    'PIE_following',     # secondary outcome
    'PIE_change',        # main predictor
    'PIE_pre',           # baseline control
    'USG_change',        # role expansion
    'AGE_post',          # development trajectory
    'MIN_post',          # playing time control
    'avg_wins_3yr',      # role security
    'POSITION_simplified' # categorical
]

print(all_candidates[model_cols].info())
print("\nMissing values:")
print(all_candidates[model_cols].isna().sum())
print("\nFirst 5 rows:")
print(all_candidates[model_cols].head().to_string())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PLAYER_NAME_post     100 non-null    str    
 1   season               100 non-null    str    
 2   PIE_diff             100 non-null    float64
 3   PIE_following        100 non-null    float64
 4   PIE_change           100 non-null    float64
 5   PIE_pre              100 non-null    float64
 6   USG_change           100 non-null    float64
 7   AGE_post             100 non-null    float64
 8   MIN_post             100 non-null    float64
 9   avg_wins_3yr         100 non-null    float64
 10  POSITION_simplified  100 non-null    str    
dtypes: float64(8), str(3)
memory usage: 8.7 KB
None

Missing values:
PLAYER_NAME_post       0
season                 0
PIE_diff               0
PIE_following          0
PIE_change             0
PIE_pre                0
USG_change             0
AGE_post    